What did I do ?
1. First I created my own Catalog (raviteja_catalog) and created Bronze, Silver , Gold and raw schemas
2. Then I created a Volume under raviteja_catalog.raw.landing and uploaded my dataset csv files 
3. To verify the volume i displayed the file using `display(dbutils.fs.ls("/Volumes/raviteja_catalog/raw/landing/"))`

In [0]:
display(dbutils.fs.ls("/Volumes/raviteja_catalog/raw/landing/"))

In [0]:
df = spark.read.option("header", "true").csv("/Volumes/raviteja_catalog/raw/landing/")
display(df)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# For file olist_orders_dataset.csv

file_path = "/Volumes/raviteja_catalog/raw/landing/olist_orders_dataset.csv"
target_table = "raviteja_catalog.bronze.orders"

df = spark.read.option("header", "true").csv(file_path)

bronze_df = (
    df.withColumn("ingest_ts", current_timestamp())
      .withColumn("source_file", input_file_name())
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# For file olist_customers_dataset.csv


file_path = "/Volumes/raviteja_catalog/raw/landing/olist_customers_dataset.csv"
target_table = "raviteja_catalog.bronze.customers"

df = spark.read.option("header", "true").csv(file_path)

bronze_df = (
    df.withColumn("ingest_ts", current_timestamp())
      .withColumn("source_file", input_file_name())
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name


# For file olist_products_dataset.csv


file_path = "/Volumes/raviteja_catalog/raw/landing/olist_products_dataset.csv"
target_table = "raviteja_catalog.bronze.products"

df = spark.read.option("header", "true").csv(file_path)

bronze_df = (
    df.withColumn("ingest_ts", current_timestamp())
      .withColumn("source_file", input_file_name())
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# For file olist_order_items_dataset.csv


file_path = "/Volumes/raviteja_catalog/raw/landing/olist_order_items_dataset.csv"
target_table = "raviteja_catalog.bronze.order_items"

df = spark.read.option("header", "true").csv(file_path)

bronze_df = (
    df.withColumn("ingest_ts", current_timestamp())
      .withColumn("source_file", input_file_name())
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# For file olist_order_payments_dataset.csv


file_path = "/Volumes/raviteja_catalog/raw/landing/olist_order_payments_dataset.csv"
target_table = "raviteja_catalog.bronze.order_payments"

df = spark.read.option("header", "true").csv(file_path)

bronze_df = (
    df.withColumn("ingest_ts", current_timestamp())
      .withColumn("source_file", input_file_name())
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# For file olist_order_reviews_dataset.csv


file_path = "/Volumes/raviteja_catalog/raw/landing/olist_order_reviews_dataset.csv"
target_table = "raviteja_catalog.bronze.order_reviews"

df = spark.read.option("header", "true").csv(file_path)

bronze_df = (
    df.withColumn("ingest_ts", current_timestamp())
      .withColumn("source_file", input_file_name())
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# For file olist_geolocation_dataset.csv


file_path = "/Volumes/raviteja_catalog/raw/landing/olist_geolocation_dataset.csv"
target_table = "raviteja_catalog.bronze.geolocation"

df = spark.read.option("header", "true").csv(file_path)

bronze_df = (
    df.withColumn("ingest_ts", current_timestamp())
      .withColumn("source_file", input_file_name())
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# For file olist_sellers_dataset.csv


file_path = "/Volumes/raviteja_catalog/raw/landing/olist_sellers_dataset.csv"
target_table = "raviteja_catalog.bronze.sellers"

df = spark.read.option("header", "true").csv(file_path)

bronze_df = (
    df.withColumn("ingest_ts", current_timestamp())
      .withColumn("source_file", input_file_name())
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# For file product_category_name_translation.csv


file_path = "/Volumes/raviteja_catalog/raw/landing/product_category_name_translation.csv"
target_table = "raviteja_catalog.bronze.product_category_translation"

df = spark.read.option("header", "true").csv(file_path)

bronze_df = (
    df.withColumn("ingest_ts", current_timestamp())
      .withColumn("source_file", input_file_name())
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

Validating the row counts after the pipeline trigger

In [0]:
%sql
SELECT COUNT(*) FROM data_sentinals.bronze.raw_customers;
-- SELECT COUNT(*) FROM data_sentinals.silver.customer_geo_silver;

In [0]:
%sql
SELECT * FROM data_sentinals.silver.delivery_order_silver LIMIT 10;
--SELECT * FROM data_sentinals.silver.order_seller_rollup_silver LIMIT 10;
--SELECT * FROM data_sentinals.silver.delivery_order_item_silver LIMIT 10;

In [0]:
%sql
SELECT delivery_status_bucket, COUNT(*) AS cnt
FROM data_sentinals.silver.delivery_order_silver
GROUP BY delivery_status_bucket
ORDER BY cnt DESC;

In [0]:
%sql
SELECT
  MIN(delivery_delta_days) AS min_delta,
  MAX(delivery_delta_days) AS max_delta,
  AVG(delivery_delta_days) AS avg_delta
FROM data_sentinals.silver.delivery_order_silver
WHERE delivery_delta_days IS NOT NULL;

In [0]:
%sql
SELECT
  customer_state,
  COUNT(*) AS orders,
  AVG(delivery_delta_days) AS avg_delta_days,
  AVG(CASE WHEN is_late = 1 THEN 1 ELSE 0 END) AS late_rate
FROM data_sentinals.silver.delivery_order_silver
WHERE delivery_delta_days IS NOT NULL
GROUP BY customer_state
ORDER BY avg_delta_days DESC;

GOLD 

In [0]:
%sql
SELECT * FROM data_sentinals.gold.gold_delivery_exec_summary;

In [0]:
%sql
SELECT *
FROM data_sentinals.gold.gold_delivery_hotspots
ORDER BY hotspot_rank
LIMIT 10;

In [0]:
%sql
SELECT *
FROM data_sentinals.gold.gold_delivery_state_month
ORDER BY order_purchase_month DESC, state_delay_rank_in_month ASC
LIMIT 20;

In [0]:
%sql
SELECT *
FROM data_sentinals.gold.gold_delivery_corridor_month
ORDER BY order_purchase_month DESC, corridor_delay_rank_in_month ASC
LIMIT 20;